# GPU & Environment Check

Run this notebook first to verify Blackwell (sm_120) compatibility.
Cells 1-3 work without PyTorch. Cells 4-5 require `install_deps.ipynb` first.

## 1. System Info

In [ ]:
import platform, sys, subprocess, shutil

print(f"OS:      {platform.system()} {platform.version()}")
print(f"Python:  {sys.version}")

nvcc = shutil.which("nvcc")
if nvcc:
    out = subprocess.check_output([nvcc, "--version"]).decode().strip()
    print(f"NVCC:    {out.splitlines()[-1]}")
else:
    print("NVCC:    not found in PATH (CUDA toolkit may not be installed)")

## 2. nvidia-smi

In [ ]:
!nvidia-smi

## 3. Parse GPU Details & Verify Blackwell

In [ ]:
import subprocess

smi = subprocess.check_output([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version,compute_cap",
    "--format=csv,noheader"
]).decode().strip()

name, vram, driver, cc = [x.strip() for x in smi.split(",")]
print(f"GPU:                {name}")
print(f"VRAM:               {vram}")
print(f"Driver:             {driver}")
print(f"Compute Capability: {cc}")

cc_major = int(cc.split(".")[0])
if cc_major >= 12:
    print(f"\n✓ Blackwell architecture confirmed (sm_{cc_major}0)")
else:
    print(f"\n✗ Expected Blackwell (cc >= 12.0), got {cc}")
    print("  PyTorch stable may work, but check CUDA compatibility.")

## 4. PyTorch CUDA Check

Run **after** `install_deps.ipynb`.

In [ ]:
try:
    import torch
    print(f"PyTorch:        {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version:   {torch.version.cuda}")
        print(f"Device:         {torch.cuda.get_device_name(0)}")
        cap = torch.cuda.get_device_capability(0)
        print(f"Capability:     {cap[0]}.{cap[1]} (sm_{cap[0]}{cap[1]}0)")
        free, total = torch.cuda.mem_get_info(0)
        print(f"VRAM:           {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if cap[0] >= 12:
            print("\n✓ PyTorch recognizes Blackwell GPU")
        else:
            print("\n✗ PyTorch does not detect sm_120 — need torch nightly with cu128")
    else:
        print("\n✗ CUDA not available. Check torch was installed with cu128 backend.")
except ImportError:
    print("PyTorch not installed yet. Run install_deps.ipynb first.")

## 5. VRAM Budget Estimates

Estimates for Qwen3-8B with 4-bit QLoRA at different LoRA ranks.

In [ ]:
try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available")

    _, total = torch.cuda.mem_get_info(0)
    total_gb = total / 1e9

    # Qwen3-8B: ~8.2B params
    # 4-bit quantized base: ~4.5 GB
    # LoRA trainable params scale with rank
    base_gb = 4.5
    configs = [
        ("r=8,  alpha=16",  0.8),
        ("r=16, alpha=32",  1.2),
        ("r=32, alpha=64",  2.0),
    ]

    print(f"Total VRAM: {total_gb:.1f} GB")
    print(f"Base model (4-bit): ~{base_gb} GB")
    print(f"{'Config':<20} {'LoRA overhead':<16} {'Total est.':<12} {'Fits?'}")
    print("-" * 60)
    for label, lora_gb in configs:
        # Add ~2 GB for optimizer states, gradients, activations
        est = base_gb + lora_gb + 2.0
        fits = "✓" if est < total_gb * 0.9 else "tight" if est < total_gb else "✗"
        print(f"{label:<20} ~{lora_gb:.1f} GB{'':<9} ~{est:.1f} GB{'':<5} {fits}")

    print(f"\nRecommendation: r=16, alpha=32 (target config)")

except (ImportError, RuntimeError) as e:
    print(f"Cannot estimate — {e}")
    print("Run install_deps.ipynb first, then re-run this cell.")

## 6. Quick Latency Baseline

In [ ]:
try:
    import torch, time
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available")

    x = torch.randn(1024, 1024, device="cuda")
    # Warmup
    for _ in range(10):
        _ = x @ x
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(100):
        _ = x @ x
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / 100

    print(f"Avg matmul (1024x1024): {elapsed*1000:.2f} ms")
    print("GPU is responsive and CUDA kernels execute correctly.")

except (ImportError, RuntimeError) as e:
    print(f"Cannot run — {e}")